# Init

In [0]:
# importing liabraries
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import *
from pyspark.sql import Window


# Rename config

In [0]:
# Rename config
RENAME_MAP = {
    "prd_id": "product_id",
    "prd_key": "product_key",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "product_start_date",
    "prd_end_dt": "product_end_date",
    "sls_prd_key": "sales_product_key",
    "cat_id": "category_id"
}

# Read data from Bronze

In [0]:
# read spark table
df = spark.table("workspace.bronze.crm_prd_info_raw")

# Transformation

## Rename column and fix column order

In [0]:
# column Rename
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)
# # column order
# df = df.select("product_id", "product_key", "product_name", "product_cost", "product_line", "product_start_date", "product_end_date", "sales_product_key", "category_id")

## Triming

In [0]:

# string data triming
for field in df.schema.fields:
    if field.dataType == StringType():
        df = df.withColumn(field.name, trim(col(field.name)))


## Normalization

In [0]:
# Normalization
df = df.withColumn(
    "product_line",
    F.when(F.upper(F.col("product_line")) == "R", "Road")
     .when(F.upper(F.col("product_line")) == "S", "Other Sales")
     .when(F.upper(F.col("product_line")) == "M", "Mountain")
     .when(F.upper(F.col("product_line")) == "T", "Touring")
     .otherwise("N/A") # Default fallback
)


## create new column: sales_product_key and category_id 

In [0]:
# create new column: cat_id by extracting first 5 character from prd_key and replace ‘-’ to ‘_’

df = df.withColumn(
    "sales_product_key",
    # Starts at position 7 and grabs characters up to the full length of the string
    F.substring(F.col("product_key"), 7, F.length(F.col("product_key")))
).withColumn(
    "category_id",
    # FIX: Using regexp_replace cleanly replaces literal strings without column errors
    F.regexp_replace(F.substring(F.col("product_key"), 1, 5), "-", "_")
)



## Replace null values with 0 in prd_cost

In [0]:
# Replace null values with 0 in prd_cost
df = df.withColumn(
    "product_cost",
    # If prd_cost is null, it falls back to a literal 0
    F.coalesce(F.col("product_cost"), F.lit(0))
)


## fix the end date

In [0]:
# fix the end date

# 1. Define window tracking spec
product_window = Window.partitionBy("product_key").orderBy("product_start_date")

# 2. Grab the next start date, subtract 1 day, and cast it safely
df = df.withColumn(
    "product_end_date",
    F.date_sub(
        F.lead(F.col("product_start_date"), 1).over(product_window), 
        1
    ).cast("date")
)


In [0]:
df.display()

# Write in Silver

In [0]:
(
    df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver.crm_prd_info")
)

In [0]:
%sql
select * from workspace.silver.crm_prd_info